# Task Dataset Build

Builds a focused species-detector dataset

- clips from target species : XC or ebird
- clips per other-present species (pooled into `non_target`)
- AudioSet ambient clips (pooled into `non_target`)

All downloads are cached: re-running the notebook only fetches what's missing.

In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
from pathlib import Path

import pyrootutils

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [33]:
LAT, LON = 48.8566, 2.3522 # PARIS
RADIUS_KM = 50

TARGET_SPECIES = [
    "Emberiza calandra",
    "Hippolais polyglotta",
    "Regulus ignicapilla",
]
TARGET_CLIPS_PER_SPECIES = 3000
NON_TARGET_CLIPS_PER_SPECIES = 250

# Area discovery (matches analysis.ipynb defaults)
MIN_AREA_RECORDINGS = 10
ANALYSIS_MAX_BIRDNET = 500
ANALYSIS_MIN_CONF = 0.85

# AudioSet ambient/anthropic noise (diversified across ~60 FOCUS_CLASSES)
AS_CLIPS_PER_CLASS = 200
AS_MAX_TOTAL_CLIPS = 15000

COLLECTION_NAME = "task_dataset"

# Discover other species present in the area

In [ ]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))
bn = await birdnet_area_analysis(
    area_df,
    max_recordings=ANALYSIS_MAX_BIRDNET,
    min_confidence=ANALYSIS_MIN_CONF,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
combined = combined_species_table(
    area_df, bn["detections"], min_recordings=MIN_AREA_RECORDINGS
)
OTHER_SPECIES = [
    sp for sp in combined["scientific_name"].tolist() if sp not in TARGET_SPECIES
]
print(f"{len(OTHER_SPECIES)} other species present in area")
OTHER_SPECIES

# Stream-download target species

In [ ]:
from building.download import download_and_process
from building.sources import XCSource, EBirdSource

with XCSource() as xc, EBirdSource() as eb:
    for sp in TARGET_SPECIES:
        await download_and_process(sp, TARGET_CLIPS_PER_SPECIES, [xc, eb])


=== [Emberiza calandra] target=6000 clips (3000/source × 2 sources)  on disk=6000 [xc=2609/3000, ebird=3391/3000] ===
[Emberiza calandra] already at target, nothing to do.

=== [Hippolais polyglotta] target=6000 clips (3000/source × 2 sources)  on disk=6000 [xc=3524/3000, ebird=2476/3000] ===
[Hippolais polyglotta] already at target, nothing to do.

=== [Regulus ignicapilla] target=6000 clips (3000/source × 2 sources)  on disk=6000 [xc=3962/3000, ebird=2038/3000] ===
[Regulus ignicapilla] already at target, nothing to do.


# Stream-download other species (250 each, pooled as non-target)

In [ ]:
with XCSource() as xc, EBirdSource() as eb:
    for sp in OTHER_SPECIES:
        await download_and_process(sp, NON_TARGET_CLIPS_PER_SPECIES, [xc, eb])


=== [Psittacula krameri] target=500 clips (250/source × 2 sources)  on disk=387 [xc=387/250, ebird=0/250] ===
[Psittacula krameri] --- Phase A: per-source quota ---
[Psittacula krameri / xc] 387/250 already on disk from this source, skipping Phase A.
[Psittacula krameri / ebird] 387/500 on disk; queued 113 of 3967 available.


[Psittacula krameri] recs:   0%|          | 0/113 [00:00<?, ?it/s]

[Psittacula krameri] clips:  77%|#######7  | 387/500 [00:00<?, ?it/s]

[Psittacula krameri] --- Phase B: top up (have 464/500) ---
[Psittacula krameri / xc] 464/500 on disk; queued 36 of 310 available.


[Psittacula krameri] recs:   0%|          | 0/36 [00:00<?, ?it/s]

...

[Erithacus rubecula / ebird] reached 500 clips.
[Erithacus rubecula] done. final 500/500 clips on disk [xc=250, ebird=250].

=== [Corvus corone corone] target=500 clips (250/source × 2 sources)  on disk=0 [xc=0/250, ebird=0/250] ===
[Corvus corone corone] --- Phase A: per-source quota ---
[Corvus corone corone / xc] no pending recordings (available=0, already processed=0, on disk=0/250).
[Corvus corone corone / ebird] 0/250 on disk; queued 44 of 44 available.


[Corvus corone corone] recs:   0%|          | 0/44 [00:00<?, ?it/s]

[Corvus corone corone] clips:   0%|          | 0/250 [00:00<?, ?it/s]

[Corvus corone corone] --- Phase B: top up (have 0/500) ---
[Corvus corone corone / xc] no pending recordings (available=0, already processed=44, on disk=0/500).
[Corvus corone corone / ebird] no pending recordings (available=44, already processed=44, on disk=0/500).
[Corvus corone corone] done. final 0/500 clips on disk [xc=0, ebird=0].

=== [Acanthis cabaret] target=500 clips (250/source × 2 sources)  on disk=250 [xc=250/250, ebird=0/250] ===
[Acanthis cabaret] --- Phase A: per-source quota ---
[Acanthis cabaret / xc] 250/250 already on disk from this source, skipping Phase A.


KeyError: "No eBird species code for 'Acanthis cabaret'"

# Stream-download AudioSet ambient clips

In [ ]:
from building.audioset import AudioSetConfig, stream_download_audioset_async

as_cfg = AudioSetConfig(
    clips_per_class=AS_CLIPS_PER_CLASS,
    max_total_clips=AS_MAX_TOTAL_CLIPS,
)
await stream_download_audioset_async(as_cfg)

[audioset] split=unbalanced_train per_class=200 global=15000 on_disk=11801 workers=6
[audioset] indexing unbalanced_train segments...
[audioset] candidates per class: Speech=936732, Male speech, man speaking=6941, Female speech, woman speaking=5057, Conversation=2140, Narration, monologue=15253, ...
[Speech] 200 clips, skipping.
[Male speech, man speaking] 200 clips, skipping.
[Female speech, woman speaking] 200 clips, skipping.
[Conversation] 200 clips, skipping.
[Narration, monologue] 200 clips, skipping.
[Children playing] no pending recordings.
[Crowd] 200 clips, skipping.
[Laughter] 200 clips, skipping.
[Applause] 200 clips, skipping.
[Cheering] 200 clips, skipping.
[Whistling] 200 clips, skipping.
[Singing] 200 clips, skipping.
[Walk, footsteps] 200 clips, skipping.
[Door] 200 clips, skipping.
[Knock] no pending recordings.
[Typing] 200 clips, skipping.
[Computer keyboard] 200 clips, skipping.
[Aircraft] 200 clips, skipping.
[Helicopter] 200 clips, skipping.
[Car] 200 clips, skip

{'speech': 200,
 'male_speech_man_speaking': 200,
 'female_speech_woman_speaking': 200,
 'conversation': 200,
 'narration_monologue': 200,
 'children_playing': 10,
 'crowd': 200,
 'laughter': 200,
 'applause': 200,
 'cheering': 200,
 'whistling': 200,
 'singing': 200,
 'walk_footsteps': 200,
 'door': 200,
 'knock': 119,
 'typing': 200,
 'computer_keyboard': 200,
 'aircraft': 200,
 'helicopter': 200,
 'car': 200,
 'motorcycle': 200,
 'truck': 200,
 'bus': 200,
 'train': 200,
 'traffic_noise_roadway_noise': 200,
 'engine': 200,
 'light_engine_high_frequency': 130,
 'chainsaw': 200,
 'lawn_mower': 200,
 'power_tool': 200,
 'drill': 200,
 'hammer': 200,
 'sawing': 200,
 'jackhammer': 133,
 'church_bell': 200,
 'bell': 200,
 'alarm': 200,
 'siren': 200,
 'dog': 200,
 'cat': 200,
 'bee_wasp_etc': 200,
 'cricket': 9,
 'insect': 200,
 'mosquito': 2,
 'fly_housefly': 2,
 'frog': 200,
 'croak': 199,
 'rain': 200,
 'rain_on_surface': 200,
 'raindrop': 200,
 'thunder': 197,
 'thunderstorm': 200,
 

# Pre-split summary

In [ ]:
from building.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _count(folder):
    return len(list(folder.glob("*.wav"))) if folder.exists() else 0


print("Target classes:")
for sp in TARGET_SPECIES:
    print(f"  {sp}: {_count(SUBSAMPLES_DIR / sp.replace(' ', '_'))}")

as_total, as_classes = 0, 0
if AUDIOSET_DIR.exists():
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            n = _count(d)
            if n:
                as_total += n
                as_classes += 1
print(f"\nAudioSet: {as_total} clips across {as_classes} classes")

other_total, other_species_present = 0, 0
for sp in OTHER_SPECIES:
    n = _count(SUBSAMPLES_DIR / sp.replace(" ", "_"))
    if n:
        other_total += n
        other_species_present += 1
print(
    f"\nOther birds: {other_total} clips "
    f"across {other_species_present}/{len(OTHER_SPECIES)} species"
)

print(f"\nBirdNET no-bird clips: {_count(BIRDNET_NO_BIRD_DIR)}")

# Per-source RMS distribution (pre-split)

Looking at raw clip levels before assembling the dataset so we can see 
the level imbalance per source. Bins of 2 dB from -70 to 0 dBFS, chosen 
from an empirical sample.

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from building.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _rms_dbfs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    rms = np.sqrt(np.mean(x**2))
    return float(20.0 * np.log10(max(rms, 1e-10)))


def _scan(files: list) -> np.ndarray:
    out: list[float] = []
    for f in tqdm(files, desc=f"scan ({len(files)})", leave=False):
        try:
            x, _ = sf.read(str(f), dtype="float32", always_2d=False)
            if x.ndim > 1:
                x = x.mean(axis=-1)
            out.append(_rms_dbfs(x))
        except Exception:
            pass
    return np.array(out)


groups: list[tuple[str, np.ndarray]] = []

# One panel per target species.
for sp in TARGET_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        groups.append((sp, _scan(sorted(folder.glob("*.wav")))))

# Other birds aggregated.
other_files: list = []
for sp in OTHER_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        other_files.extend(folder.glob("*.wav"))
groups.append((f"Other birds ({len(OTHER_SPECIES)} sp.)", _scan(other_files)))

# BirdNET no-bird clips.
if BIRDNET_NO_BIRD_DIR.exists():
    groups.append(
        ("BirdNET no-bird", _scan(sorted(BIRDNET_NO_BIRD_DIR.glob("*.wav"))))
    )

# AudioSet aggregated across focus classes.
if AUDIOSET_DIR.exists():
    as_files: list = []
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            as_files.extend(d.glob("*.wav"))
    groups.append(("AudioSet (all)", _scan(as_files)))

print(
    f"{'group':<32s} {'n':>5s}  {'q05':>6s} {'q50':>6s} {'q95':>6s}  "
    f"{'min':>6s} {'max':>6s}"
)
for label, a in groups:
    if len(a) == 0:
        continue
    print(
        f"{label:<32s} {len(a):>5d}  "
        f"{np.quantile(a, 0.05):>+6.1f} {np.median(a):>+6.1f} {np.quantile(a, 0.95):>+6.1f}  "
        f"{a.min():>+6.1f} {a.max():>+6.1f}"
    )

bins = np.arange(-70, 1, 2)
n = len(groups)
fig, axes = plt.subplots(n, 1, figsize=(10, max(2.0, 1.6 * n)), sharex=True)
if n == 1:
    axes = [axes]
for ax, (label, a) in zip(axes, groups):
    if len(a):
        ax.hist(a, bins=bins, color="steelblue", edgecolor="black", alpha=0.85)
        ax.axvline(
            np.median(a), color="red", lw=1, ls="--", label=f"median {np.median(a):+.1f}"
        )
        ax.legend(loc="upper left", fontsize=8, frameon=False)
    ax.set_ylabel(label, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("RMS (dBFS)")
fig.suptitle("Per-source RMS distribution (all data, pre-split)")
plt.tight_layout()
plt.show()


# Assemble train/val/test

In [ ]:
from building.dataset import build_task_dataset

dataset_path = build_task_dataset(
    COLLECTION_NAME,
    target_species=TARGET_SPECIES,
    non_target_species=OTHER_SPECIES,
)
print(dataset_path)

Non-target pool: 17128 clips (8288 audioset + 8288 xc_other + 552 birdnet_no_bird)
  audioset:         take 8288 of 11801 available
  xc_other:         take 8288 of 8288 available
  birdnet_no_bird:  take 552 of 552 available
Processing Emberiza_calandra with 2609 clips
Training: 1820, Validation: 372, Testing: 417
Copied 1820 clips from Emberiza_calandra to training
Copied 372 clips from Emberiza_calandra to validation
Copied 417 clips from Emberiza_calandra to testing
Processing Hippolais_polyglotta with 3000 clips
Training: 2100, Validation: 446, Testing: 454
Copied 2100 clips from Hippolais_polyglotta to training
Copied 446 clips from Hippolais_polyglotta to validation
Copied 454 clips from Hippolais_polyglotta to testing
Processing Regulus_ignicapilla with 3000 clips
Training: 2100, Validation: 450, Testing: 450
Copied 2100 clips from Regulus_ignicapilla to training
Copied 450 clips from Regulus_ignicapilla to validation
Copied 450 clips from Regulus_ignicapilla to testing
Process